In [ ]:
# Install required packages (auto-skipped if already installed)
import importlib
if importlib.util.find_spec('qiskit') is None:
    !pip install -q qiskit qiskit-aer qiskit-ibm-runtime pylatexenc matplotlib
else:
    print("\u2713 Packages already installed")

# To run on real quantum hardware, uncomment and fill in your credentials:
# from qiskit_ibm_runtime import QiskitRuntimeService
# QiskitRuntimeService.save_account(
#     token="<your-api-key>",
#     instance="<your-crn>",
#     overwrite=True
# )

# Peringkat Transpiler

<details>
<summary><b>Versi pakej</b></summary>

Kod pada halaman ini dibangunkan menggunakan keperluan berikut.
Kami mengesyorkan penggunaan versi ini atau yang lebih baharu.

```
qiskit[all]~=2.3.0
qiskit-ibm-runtime~=0.43.1
```
</details>
Halaman ini menerangkan peringkat saluran transpilasi prabina dalam Qiskit SDK. Terdapat enam peringkat:

1. `init`
2. `layout`
3. `routing`
4. `translation`
5. `optimization`
6. `scheduling`

Fungsi [`generate_preset_pass_manager`](https://docs.quantum.ibm.com/api/qiskit/qiskit.transpiler.generate_preset_pass_manager#qiskit.transpiler.generate_preset_pass_manager) mencipta [pengurus laluan berperingkat](https://docs.quantum.ibm.com/api/qiskit/qiskit.transpiler.StagedPassManager) pratetap yang terdiri daripada peringkat-peringkat ini. Laluan khusus yang membentuk setiap peringkat bergantung kepada argumen yang dihantar kepada `generate_preset_pass_manager`. `optimization_level` ialah argumen kedudukan yang mesti ditentukan; ia adalah integer yang boleh bernilai 0, 1, 2, atau 3. Nilai yang lebih tinggi menunjukkan pengoptimuman yang lebih berat tetapi lebih mahal (lihat [Tetapan lalai dan pilihan konfigurasi transpilasi](defaults-and-configuration-options)).

Cara yang disyorkan untuk mentranspilasi Circuit ialah dengan mencipta pengurus laluan berperingkat pratetap dan kemudian menjalankan pengurus laluan tersebut pada Circuit, seperti yang diterangkan dalam [Transpilasi dengan pengurus laluan](transpile-with-pass-managers). Walau bagaimanapun, alternatif yang lebih mudah tetapi kurang boleh disesuaikan ialah menggunakan fungsi [`transpile`](https://docs.quantum.ibm.com/api/qiskit/compiler#qiskit.compiler.transpile). Fungsi ini menerima Circuit secara langsung sebagai argumen. Seperti `generate_preset_pass_manager`, laluan Transpiler khusus yang digunakan bergantung kepada argumen, seperti `optimization_level`, yang dihantar kepada `transpile`. Malah, secara dalaman fungsi `transpile` memanggil `generate_preset_pass_manager` untuk mencipta pengurus laluan berperingkat pratetap dan menjalankannya pada Circuit.
## Peringkat Init
Peringkat pertama ini melakukan sangat sedikit secara lalai dan terutamanya berguna jika anda ingin menyertakan pengoptimuman awal anda sendiri. Memandangkan kebanyakan algoritma susun atur dan penghalaan hanya direka bentuk untuk berfungsi dengan Gate satu- dan dua-Qubit, peringkat ini juga digunakan untuk menterjemah sebarang Gate yang beroperasi pada lebih daripada dua Qubit, menjadi Gate yang hanya beroperasi pada satu atau dua Qubit.

Untuk maklumat lanjut tentang melaksanakan pengoptimuman awal anda sendiri untuk peringkat ini, lihat bahagian tentang plugin dan penyesuaian pengurus laluan.
## Peringkat Layout
Peringkat seterusnya melibatkan susun atur atau ketersambungan Backend yang akan dihantar Circuit. Secara umum, Circuit kuantum adalah entiti abstrak yang Qubit-nya merupakan representasi "maya" atau "logik" daripada Qubit sebenar yang digunakan dalam pengiraan. Untuk melaksanakan urutan Gate, pemetaan satu-ke-satu daripada Qubit "maya" ke Qubit "fizikal" dalam peranti kuantum sebenar diperlukan. Pemetaan ini disimpan sebagai objek `Layout` dan merupakan sebahagian daripada kekangan yang ditakrifkan dalam [seni bina set arahan (ISA)](./transpile#instruction-set-architecture) Backend.

![Imej ini menggambarkan Qubit yang dipetakan daripada representasi wayar ke gambar rajah yang mewakili cara Qubit disambungkan pada QPU.](../docs/images/guides/transpiler-stages/layout-mapping.avif "Pemetaan Qubit")

Pilihan pemetaan amat penting untuk meminimumkan bilangan operasi SWAP yang diperlukan untuk memetakan Circuit input kepada topologi peranti dan memastikan Qubit yang paling baik dikalibrasi digunakan. Disebabkan kepentingan peringkat ini, pengurus laluan pratetap mencuba beberapa kaedah berbeza untuk mencari susun atur terbaik. Biasanya ini melibatkan dua langkah: pertama, cuba mencari susun atur "sempurna" (susun atur yang tidak memerlukan sebarang operasi SWAP), dan kemudian, laluan heuristik yang cuba mencari susun atur terbaik untuk digunakan jika susun atur sempurna tidak dapat dijumpai. Terdapat dua `Laluan` yang biasanya digunakan untuk langkah pertama ini:

- `TrivialLayout`: Memetakan setiap Qubit maya secara naif ke Qubit fizikal bernombor sama pada peranti (cth., [`0`,`1`,`1`,`3`] -> [`0`,`1`,`1`,`3`]). Ini adalah kelakuan sejarah yang hanya digunakan dalam `optimzation_level=1` untuk cuba mencari susun atur sempurna. Jika gagal, `VF2Layout` dicuba seterusnya.
- `VF2Layout`: Ini ialah `AnalysisPass` yang memilih susun atur ideal dengan menganggap peringkat ini sebagai masalah isomorfisme subgraf, diselesaikan oleh algoritma VF2++. Jika lebih daripada satu susun atur dijumpai, heuristik pemarkahan dijalankan untuk memilih pemetaan dengan ralat purata terendah.

Kemudian untuk peringkat heuristik, dua laluan digunakan secara lalai:

- `DenseLayout`: Mencari sub-graf peranti dengan ketersambungan tertinggi dan yang mempunyai bilangan Qubit yang sama dengan Circuit (digunakan untuk tahap pengoptimuman 1 jika terdapat operasi aliran kawalan (seperti IfElseOp) dalam Circuit).
- `SabreLayout`: Laluan ini memilih susun atur dengan bermula dari susun atur rawak awal dan berulang kali menjalankan algoritma `SabreSwap`. Laluan ini hanya digunakan dalam tahap pengoptimuman 1, 2, dan 3 jika susun atur sempurna tidak dijumpai melalui laluan `VF2Layout`. Untuk butiran lanjut tentang algoritma ini, rujuk kertas [arXiv:1809.02573.](https://arxiv.org/abs/1809.02573)
## Peringkat Routing
Untuk melaksanakan Gate dua-Qubit antara Qubit yang tidak disambungkan secara langsung pada peranti kuantum, satu atau lebih Gate SWAP mesti disisipkan ke dalam Circuit untuk menggerakkan keadaan Qubit sehingga ia bersebelahan pada peta Gate peranti. Setiap Gate SWAP mewakili operasi yang mahal dan berderau untuk dilakukan. Oleh itu, mencari bilangan minimum Gate SWAP yang diperlukan untuk memetakan Circuit kepada peranti tertentu adalah langkah penting dalam proses transpilasi. Untuk kecekapan, peringkat ini biasanya dikira bersama peringkat Layout secara lalai, tetapi keduanya adalah berbeza secara logik. Peringkat *Layout* memilih Qubit perkakasan yang hendak digunakan, manakala peringkat *Routing* menyisipkan jumlah Gate SWAP yang sesuai untuk melaksanakan Circuit menggunakan susun atur yang dipilih.

Walau bagaimanapun, mencari pemetaan SWAP yang optimum adalah sukar. Malah, ia adalah masalah NP-sukar, dan oleh itu terlalu mahal untuk dikira untuk semua peranti kuantum dan Circuit input kecuali yang paling kecil. Untuk mengatasi ini, Qiskit menggunakan algoritma heuristik stokastik yang dipanggil `SabreSwap` untuk mengira pemetaan SWAP yang baik, tetapi tidak semestinya optimum. Penggunaan kaedah stokastik bermakna Circuit yang dijana tidak dijamin sama sepanjang penggunaan berulang. Sesungguhnya, menjalankan Circuit yang sama berulang kali menghasilkan taburan kedalaman Circuit dan bilangan Gate pada output. Atas sebab inilah banyak pengguna memilih untuk menjalankan fungsi penghalaan (atau keseluruhan `StagedPassManager`) berkali-kali dan memilih Circuit dengan kedalaman terendah daripada taburan output.

Sebagai contoh, mari kita ambil Circuit GHZ 15-Qubit yang dilaksanakan 100 kali, menggunakan `initial_layout` yang "buruk" (terputus).

In [1]:
import matplotlib.pyplot as plt
from qiskit import QuantumCircuit
from qiskit_ibm_runtime.fake_provider import FakeAuckland, FakeWashingtonV2
from qiskit.transpiler import generate_preset_pass_manager

backend = FakeAuckland()

ghz = QuantumCircuit(15)
ghz.h(0)
ghz.cx(0, range(1, 15))

depths = []
for seed in range(100):
    pass_manager = generate_preset_pass_manager(
        optimization_level=1,
        backend=backend,
        layout_method="trivial",  # Fixed layout mapped in circuit order
        seed_transpiler=seed,  # For reproducible results
    )
    depths.append(pass_manager.run(ghz).depth())

plt.figure(figsize=(8, 6))
plt.hist(depths, align="left", color="#AC557C")
plt.xlabel("Depth", fontsize=14)
plt.ylabel("Counts", fontsize=14)

Text(0, 0.5, 'Counts')

<Image src="../docs/images/guides/transpiler-stages/extracted-outputs/358cfb50-02fc-48f2-a4ec-657837e0c304-1.svg" alt="Output of the previous code cell" />

This wide distribution demonstrates how difficult it is for the SWAP mapper to compute the best mapping.  To gain some insight, let's look at both the circuit being executed as well as the qubits that were chosen on the hardware.

In [2]:
ghz.draw("mpl", idle_wires=False)

<Image src="../docs/images/guides/transpiler-stages/extracted-outputs/bb3b8c1f-69fd-4e0c-9b78-9ee67f3361bb-0.svg" alt="Output of the previous code cell" />

In [3]:
from qiskit.visualization import plot_circuit_layout

# Plot the hardware graph and indicate which hardware qubits were chosen to run the circuit
transpiled_circ = pass_manager.run(ghz)
plot_circuit_layout(transpiled_circ, backend)

<Image src="../docs/images/guides/transpiler-stages/extracted-outputs/e4cd49ef-5b3e-4ee0-82f8-c83e1f0ae755-0.svg" alt="Output of the previous code cell" />

As you can see, this circuit has to execute a two-qubit gate between qubits 0 and 14, which are very far apart on the connectivity graph.  Running this circuit thus requires inserting SWAP gates to execute all of the two-qubit gates using the `SabreSwap` pass.

Note also that the `SabreSwap` algorithm is different from the larger `SabreLayout` method in the previous stage.  By default, `SabreLayout` runs both layout and routing, and returns the transformed circuit.  This is done for a few particular technical reasons specified in the pass's [API reference page](../api/qiskit/qiskit.transpiler.passes.SabreLayout).

## Translation stage

When writing a quantum circuit, you are free to use any quantum gate (unitary operation) that you like, along with a collection of non-gate operations such as qubit measurement or reset instructions.  However, most quantum devices only natively support a handful of quantum gate and non-gate operations.  These native gates are part of the definition of a target's [ISA](./transpile#instruction-set-architecture) and this stage of the preset `PassManagers`  translates (or *unrolls*) the gates specified in a circuit to the native basis gates of a specified backend.  This is an important step, as it allows the circuit to be executed by the backend, but typically leads to an increase in the depth and number of gates.

Two special cases are especially important to highlight, and help illustrate what this stage does.

1. If a SWAP gate is not a native gate to the target backend, this requires three CNOT gates:

In [4]:
print("native gates:" + str(sorted(backend.operation_names)))
qc = QuantumCircuit(2)
qc.swap(0, 1)
qc.decompose().draw("mpl")

native gates:['cx', 'delay', 'for_loop', 'id', 'if_else', 'measure', 'reset', 'rz', 'switch_case', 'sx', 'x']


<Image src="../docs/images/guides/transpiler-stages/extracted-outputs/31fcf8b6-4f3a-460e-90fd-446813bd4f28-1.svg" alt="Output of the previous code cell" />

![Output of the previous code cell](../docs/images/guides/transpiler-stages/extracted-outputs/e4cd49ef-5b3e-4ee0-82f8-c83e1f0ae755-0.svg)

Seperti yang dapat anda lihat, Circuit ini perlu melaksanakan Gate dua-Qubit antara Qubit 0 dan 14, yang sangat jauh pada graf ketersambungan. Menjalankan Circuit ini dengan itu memerlukan penyisipan Gate SWAP untuk melaksanakan semua Gate dua-Qubit menggunakan laluan `SabreSwap`.

Perhatikan juga bahawa algoritma `SabreSwap` berbeza daripada kaedah `SabreLayout` yang lebih besar dalam peringkat sebelumnya. Secara lalai, `SabreLayout` menjalankan kedua-dua susun atur dan penghalaan, dan mengembalikan Circuit yang ditransformasi. Ini dilakukan untuk beberapa sebab teknikal khusus yang dinyatakan dalam [halaman rujukan API](../api/qiskit/qiskit.transpiler.passes.SabreLayout) laluan tersebut.
## Peringkat Translation
Apabila menulis Circuit kuantum, anda bebas menggunakan sebarang Gate kuantum (operasi uniter) yang anda suka, bersama koleksi operasi bukan-Gate seperti arahan pengukuran atau set semula Qubit. Walau bagaimanapun, kebanyakan peranti kuantum hanya menyokong segelintir operasi Gate kuantum dan bukan-Gate secara natif. Gate natif ini merupakan sebahagian daripada takrifan [ISA](./transpile#instruction-set-architecture) sasaran dan peringkat `PassManager` pratetap ini menterjemah (atau *membuka gulungan*) Gate yang ditentukan dalam Circuit ke Gate asas natif Backend yang ditentukan. Ini adalah langkah penting, kerana ia membolehkan Circuit dilaksanakan oleh Backend, tetapi biasanya menyebabkan peningkatan dalam kedalaman dan bilangan Gate.

Dua kes khas amat penting untuk ditonjolkan, dan membantu menggambarkan apa yang dilakukan oleh peringkat ini.

1. Jika Gate SWAP bukan Gate natif pada Backend sasaran, ini memerlukan tiga Gate CNOT:

In [5]:
qc = QuantumCircuit(3)
qc.ccx(0, 1, 2)
qc.decompose().draw("mpl")

<Image src="../docs/images/guides/transpiler-stages/extracted-outputs/8a2c1913-3324-44b0-859e-d7fd348161c3-0.svg" alt="Output of the previous code cell" />

For every Toffoli gate in a quantum circuit, the hardware may execute up to six CNOT gates and a handful of single-qubit gates.  This example demonstrates that any algorithm making use of multiple Toffoli gates will end up as a circuit with large depth and will therefore be appreciably affected by noise.

## Optimization stage

This stage centers around decomposing quantum circuits into the basis gate set of the target device, and must fight against the increased depth from the layout and routing stages.  Fortunately, there are many routines for optimizing circuits by either combining or eliminating gates.  In some cases, these methods are so effective that the output circuits have lower depth than the inputs, even after layout and routing to the hardware topology.  In other cases, not much can be done, and the computation may be difficult to perform on noisy devices.  This stage is where the various optimization levels begin to differ.

- For `optimization_level=1`, this stage prepares [`Optimize1qGatesDecomposition`](../api/qiskit/qiskit.transpiler.passes.Optimize1qGatesDecomposition) and [`CXCancellation`](/docs/api/qiskit/1.4/qiskit.transpiler.passes.CXCancellation), which combine chains of single-qubit gates and cancel any back-to-back CNOT gates.
- For `optimization_level=2`, this stage uses the [`CommutativeCancellation`](../api/qiskit/qiskit.transpiler.passes.CommutativeCancellation) pass instead of `CXCancellation`, which removes redundant gates by exploiting commutation relations.
- For `optimization_level=3`, this stage prepares the following passes:
  - [`Collect2qBlocks`](../api/qiskit/qiskit.transpiler.passes.Collect2qBlocks)
  - [`ConsolidateBlocks`](../api/qiskit/qiskit.transpiler.passes.ConsolidateBlocks)
  - [`UnitarySynthesis`](../api/qiskit/qiskit.transpiler.passes.UnitarySynthesis)
  - [`Optimize1qGateDecomposition`](../api/qiskit/qiskit.transpiler.passes.Optimize1qGatesDecomposition)
  - [`CommutativeCancellation`](../api/qiskit/qiskit.transpiler.passes.CommutativeCancellation)


Additionally, this stage also executes a few final checks to make sure that all instructions in the circuit are composed of the basis gates available on the target backend.

The example below using a GHZ state demonstrates the effects of different optimization level settings on circuit depth and gate count.

<Admonition type="note">
  The transpilation output varies due to the stochastic SWAP mapper. Therefore, the numbers below will likely change each time you run the code.
</Admonition>

![15-qubit GHZ state](../docs/images/guides/transpiler-stages/transpiler-11.avif "15-qubit GHZ state before transpilation")

The following code constructs a 15-qubit GHZ state and compares the `optimization_levels` of transpilation in terms of resulting circuit depth, gate counts, and multi-qubit gate counts.

In [6]:
ghz = QuantumCircuit(15)
ghz.h(0)
ghz.cx(0, range(1, 15))

depths = []
gate_counts = []
multiqubit_gate_counts = []
levels = [str(x) for x in range(4)]
for level in range(4):
    pass_manager = generate_preset_pass_manager(
        optimization_level=level,
        backend=backend,
        seed_transpiler=1234,
    )
    circ = pass_manager.run(ghz)
    depths.append(circ.depth())
    gate_counts.append(sum(circ.count_ops().values()))
    multiqubit_gate_counts.append(circ.count_ops()["cx"])

fig, (ax1, ax2) = plt.subplots(2, 1)
ax1.bar(levels, depths, label="Depth")
ax1.set_xlabel("Optimization Level")
ax1.set_ylabel("Depth")
ax1.set_title("Output Circuit Depth")
ax2.bar(levels, gate_counts, label="Number of Circuit Operations")
ax2.bar(levels, multiqubit_gate_counts, label="Number of CX gates")
ax2.set_xlabel("Optimization Level")
ax2.set_ylabel("Number of gates")
ax2.legend()
ax2.set_title("Number of output circuit gates")
fig.tight_layout()
plt.show()

<Image src="../docs/images/guides/transpiler-stages/extracted-outputs/2507de9c-1b94-4d56-b5a6-0b2bb1719a80-0.svg" alt="Output of the previous code cell" />

![Output of the previous code cell](../docs/images/guides/transpiler-stages/extracted-outputs/31fcf8b6-4f3a-460e-90fd-446813bd4f28-1.svg)

Sebagai hasil tiga Gate CNOT, SWAP adalah operasi yang mahal untuk dilakukan pada peranti kuantum berderau. Walau bagaimanapun, operasi sedemikian biasanya diperlukan untuk membenamkan Circuit ke dalam ketersambungan Gate terhad banyak peranti. Oleh itu, meminimumkan bilangan Gate SWAP dalam Circuit adalah matlamat utama dalam proses transpilasi.

2. Gate Toffoli, atau not-terkawal-terkawal (`ccx`), ialah Gate tiga-Qubit. Memandangkan set Gate asas kita hanya termasuk Gate satu- dan dua-Qubit, operasi ini mesti diuraikan. Walau bagaimanapun, ia agak mahal:

In [7]:
ghz = QuantumCircuit(5)
ghz.h(0)
ghz.cx(0, range(1, 5))


# Use fake backend
backend = FakeWashingtonV2()

# Run with optimization level 3 and 'asap' scheduling pass
pass_manager = generate_preset_pass_manager(
    optimization_level=3,
    backend=backend,
    scheduling_method="asap",
    seed_transpiler=1234,
)


circ = pass_manager.run(ghz)
circ.draw(output="mpl", idle_wires=False)

<Image src="../docs/images/guides/transpiler-stages/extracted-outputs/4339deb5-4947-493b-8940-e68c91311631-0.svg" alt="Output of the previous code cell" />

![Output of the previous code cell](../docs/images/guides/transpiler-stages/extracted-outputs/8a2c1913-3324-44b0-859e-d7fd348161c3-0.svg)

Untuk setiap Gate Toffoli dalam Circuit kuantum, perkakasan mungkin melaksanakan sehingga enam Gate CNOT dan segelintir Gate satu-Qubit. Contoh ini menunjukkan bahawa sebarang algoritma yang menggunakan berbilang Gate Toffoli akan berakhir sebagai Circuit dengan kedalaman besar dan oleh itu akan terjejas dengan ketara oleh hingar.
## Peringkat Optimization
Peringkat ini berpusat pada penguraian Circuit kuantum ke dalam set Gate asas peranti sasaran, dan mesti berjuang menentang peningkatan kedalaman daripada peringkat susun atur dan penghalaan. Syukurlah, terdapat banyak rutin untuk mengoptimumkan Circuit sama ada dengan menggabungkan atau menghapuskan Gate. Dalam sesetengah kes, kaedah-kaedah ini begitu berkesan sehingga Circuit output mempunyai kedalaman yang lebih rendah daripada input, walaupun selepas susun atur dan penghalaan ke topologi perkakasan. Dalam kes lain, tidak banyak yang boleh dilakukan, dan pengiraan mungkin sukar dilakukan pada peranti berderau. Peringkat inilah di mana pelbagai tahap pengoptimuman mula berbeza.

- Untuk `optimization_level=1`, peringkat ini menyediakan [`Optimize1qGatesDecomposition`](../api/qiskit/qiskit.transpiler.passes.Optimize1qGatesDecomposition) dan [`CXCancellation`](https://docs.quantum.ibm.com/api/qiskit/1.4/qiskit.transpiler.passes.CXCancellation), yang menggabungkan rantaian Gate satu-Qubit dan membatalkan sebarang Gate CNOT berturut-turut.
- Untuk `optimization_level=2`, peringkat ini menggunakan laluan [`CommutativeCancellation`](../api/qiskit/qiskit.transpiler.passes.CommutativeCancellation) sebagai ganti `CXCancellation`, yang menghapus Gate berlebihan dengan mengeksploitasi hubungan komutasi.
- Untuk `optimization_level=3`, peringkat ini menyediakan laluan-laluan berikut:
  - [`Collect2qBlocks`](../api/qiskit/qiskit.transpiler.passes.Collect2qBlocks)
  - [`ConsolidateBlocks`](../api/qiskit/qiskit.transpiler.passes.ConsolidateBlocks)
  - [`UnitarySynthesis`](../api/qiskit/qiskit.transpiler.passes.UnitarySynthesis)
  - [`Optimize1qGateDecomposition`](../api/qiskit/qiskit.transpiler.passes.Optimize1qGatesDecomposition)
  - [`CommutativeCancellation`](../api/qiskit/qiskit.transpiler.passes.CommutativeCancellation)

Selain itu, peringkat ini juga melaksanakan beberapa pemeriksaan akhir untuk memastikan semua arahan dalam Circuit terdiri daripada Gate asas yang tersedia pada Backend sasaran.

Contoh di bawah menggunakan keadaan GHZ menunjukkan kesan tetapan tahap pengoptimuman yang berbeza terhadap kedalaman Circuit dan bilangan Gate.

> **Note:** Output transpilasi berbeza disebabkan pemeta SWAP stokastik. Oleh itu, nombor-nombor di bawah kemungkinan akan berubah setiap kali anda menjalankan kod.

![Keadaan GHZ 15-Qubit](../docs/images/guides/transpiler-stages/transpiler-11.avif "Keadaan GHZ 15-Qubit sebelum transpilasi")

Kod berikut membina keadaan GHZ 15-Qubit dan membandingkan `optimization_levels` transpilasi dari segi kedalaman Circuit, bilangan Gate, dan bilangan Gate berbilang-Qubit yang terhasil.